In [ ]:
import sys
import os
from pathlib import Path
import multiprocessing as mp

# 1. Path Resolution
# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))
if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/haman/mf-temporary')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    repo_path = Path('/home/pavel/academia/wintermute/mf-temporary')

# 2. The Great MPI Blindfold
keys_to_delete = [k for k in os.environ if k.startswith('SLURM') or k.startswith('OMPI_') or k.startswith('PRTE_') or 'HYDRA' in k]
for key in keys_to_delete:
    del os.environ[key]

# Prevent CPU hogging by the C++ backend
os.environ["OMP_NUM_THREADS"] = "1" 

# 3. Safe Multiprocessing
try:
    mp.set_start_method('spawn', force=True)
    print("Multiprocessing context set to:", mp.get_start_method())
except RuntimeError:
    pass

print(f"Loaded paths securely. MPI blindfolded. Ready for testing!")

In [ ]:
from importlib import reload

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from codes.controller.config import load_workflow_config
from codes.stimuli.loader import load_stimuli_config
from codes.network_params.loader import load_network_parameters

from codes.neuron_simulation import run_neuron_simulation_workflow
from codes.transfer_function import run_tf_fitting_workflow
from codes.mf_simulation import run_mf_simulation_workflow
from codes.snn_simulation import run_snn_simulation_workflow

from codes.plotting import fig_plots
import codes.stimuli as stim
from pydantic import BaseModel

project_path = repo_path / "projects" / "00_drafts_tests"
os.chdir(project_path)

In [ ]:
network_params = load_network_parameters(project_path / "params" / "network_params_new.yaml")
sim_params = load_workflow_config(project_path / "params" / "workflow_params_new.yaml")
stimuli_config = load_stimuli_config(project_path / "params" / "stimuli.yaml")


In [ ]:
# 1. neuron simulation
neuron_results = run_neuron_simulation_workflow(sim_params.neuron_simulation, network_params)

In [ ]:
fig_name = f"neuron_activity.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_neuron_activity(
    neuron_results, 
    common_params={}, 
    fig_params={
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity"
    }
)
display(Image(filename=str(fig_path)))

fig_name = f"std_neuron_activity.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    {"exc_neuron": [],
    "inh_neuron": []
    }, 
    common_params={
        'labels' : [],
        'linestyles' : [],
        # 'xlim' : (0,7),
        'ylim' : (0, 60),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity (STD)"
    })
display(Image(filename=str(fig_path)))

In [ ]:
# 2. loop through mf_models and run tf fitting
tf_results_dict = {}
tf_results_legacy = {
    "exc_neuron": [],
    "inh_neuron": []
}
for mf_model_name, mf_sim_params in sim_params.mf_models.items():
    tf_results = run_tf_fitting_workflow(mf_sim_params.transfer_function, network_params, neuron_results)
    tf_results_dict[mf_model_name] = tf_results
    tf_results_legacy["exc_neuron"].append(tf_results["exc_neuron"])
    tf_results_legacy["inh_neuron"].append(tf_results["inh_neuron"])

In [ ]:
fig_name = f"std_neuron_activity.png"
fig_path = project_path / "imgs" / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results,
    tf_results_legacy,
    common_params={
        'labels' : list(sim_params.mf_models.keys()),
        'linestyles' : ["--"],
        # 'xlim' : (0,7),
        'ylim' : (0, 60),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
        'title': f"Neuron Activity (STD)"
    })
display(Image(filename=str(fig_path)))

In [ ]:
# 3. loop through stimuli and run snn_simulation and mf_simulation, then plot results

snn_results = run_snn_simulation_workflow(sim_params.snn_simulation, network_params, stimuli_config)
mf_results_dict = {}
for mf_model_name, mf_sim_params in sim_params.mf_models.items():
    mf_results = run_mf_simulation_workflow(mf_sim_params, network_params, stimuli_config)
    mf_results_dict[mf_model_name] = mf_results



In [ ]:
for stim_name in stimuli_config.keys():

    fig_name = f"activity_{stim_name}.png"
    fig_path = project_path / "imgs" / fig_name
    fig_plots.fig_activity_comparison(
        snn_results,
        mf_results,
        common_params={
            'labels' : ["SNN", "MF"],
            'linestyles' : ["-", "--"],
            # 'xlim' : (0,7),
            'ylim' : (0, 60),
        }, 
        fig_params={
            'fontsize': 14,
            'figsize': (15, 10),  # width, height
            'tight_layout': True,
            'savefig': True,
            'savefig_path': fig_path,
            'title': f"Activity Comparison - {stim_name}"
        })
    display(Image(filename=str(fig_path)))

In [ ]:
for stimulus_name in stimuli_config.keys():
    fig_plots.fig_full_network_overview_together(
        snn_results[stimulus_name], 
        [],
        common_params={
            'xmargin': 0.0,
            'ymargin': 0.0,
            'labels': ['SNN'] + [],
            'legend': {'loc': 'upper left'},
            'xlim' : (0, snn_results[stimulus_name].times()[-1])
        },
        fig_params={
            'figsize': (20, 10),  # width, height
            'tight_layout': True,
            'savefig': True,
            'savefig_path': project_path / "imgs" / f"{stimulus_name}_Full_network_overview_together.png",
            'title' : f"Network overview for stimulus: '{stimulus_name}' and drive rate: {stimuli_config[stimulus_name].drive_rate} Hz",
        }
    )

    display(Image(filename=str(project_path / "imgs" / f"{stimulus_name}_Full_network_overview_together.png")))
